# CRS v5 Full Run (All 80 logs: P0 / P1 / P2 / P3 with ResNet-50 Zoom + YOLO Snap + Auto Time Budget Probe)

In [ ]:
!pip install -q "transformers>=4.57" peft pyyaml certifi matplotlib ultralytics
import os, subprocess, glob, json, math, torch
ROOT = os.getcwd() if os.path.exists('run.py') else os.path.abspath('crs_road')
os.chdir(ROOT)
LOGS, FR, DATA = '../CRS_Logs', '../av2_frames', '../data'
# Slow/mid GPU preset (T4/3060/4060): --dtype fp16 --grad_ckpt --img_side 336 --lora_r 8, BATCH=8
# Fast GPU preset (A6000/A100 48GB):  --dtype bf16 --img_side 448 --lora_r 16, BATCH=16
VLM = '--dtype bf16 --img_side 336 --lora_r 8 --grad_ckpt --cnn_arch resnet50'
BUDGET_HOURS = 10.0   # total VLM training hours across P1 + P2 + P3
BATCH = 8
print(torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory // 2**30, 'GB')

In [ ]:
!python download_logs.py --out {LOGS}

In [ ]:
from concurrent.futures import ThreadPoolExecutor
ids = sorted(os.path.basename(f)[:-5] for f in glob.glob(f'{LOGS}/annotations/*.json'))
def go(lid):
    if os.path.exists(f'{FR}/{lid}/frames.json'): return lid, 'skip'
    r = subprocess.run(['python', 'fetch_av2_frames.py', '--log', lid, '--meta', f'{LOGS}/metadata', '--out', FR], capture_output=True, text=True)
    return lid, (r.stdout.strip().splitlines() or ['?'])[-1] + r.stderr[-150:]
with ThreadPoolExecutor(4) as ex:
    for lid, m in ex.map(go, ids): print(lid[:8], m)

In [ ]:
!python run.py real --graphs {LOGS}/annotations --meta {LOGS}/metadata --frames {FR} --out {DATA} --max_per_log 300 --split 50,15,15
!python run.py check --qa {DATA}
!python run.py detect --scenes {DATA}/scenes --model yolo11n.pt
!python check_real.py --scenes {DATA}/scenes --out ../overlays --n 5

In [ ]:
!python run.py cropcache --scenes {DATA}/scenes --out {DATA}/crops
!python run.py cnn --arch resnet50 --cache {DATA}/crops --fold 0 --nfolds 2 --epochs 6 --bs 64 --out ../ck/c0_f0.pt
!python run.py cnn --arch resnet50 --cache {DATA}/crops --fold 1 --nfolds 2 --epochs 6 --bs 64 --out ../ck/c0_f1.pt

## 5. Probe GPU speed & compute step count for `BUDGET_HOURS`

In [ ]:
!python run.py train --variant p2 --probe --scenes {DATA}/scenes --qa_train {DATA}/qa_train.jsonl --out ../runs/probe {VLM}
sec_per_sample = json.load(open('../runs/probe/probe.json'))['sec_per_sample']
n_train = sum(1 for _ in open(f'{DATA}/qa_train.jsonl'))
max_steps_2ep = math.ceil(2.0 * n_train / BATCH)
# P1 (~0.8x) + P2 prefix (0.7x) + P2b (0.3x) + P3 (0.45x) ~= 2.25x of one P2 run
steps_budget = int((BUDGET_HOURS * 3600) / (2.25 * sec_per_sample * BATCH))
STEPS = max(200, min(max_steps_2ep, steps_budget))
print(f"probe={sec_per_sample:.3f}s/sample | n_train={n_train} | STEPS={STEPS} (~{2.25*STEPS*BATCH*sec_per_sample/3600:.1f}h total)")

In [ ]:
EV_STEP = max(50, STEPS // 8)
TR = f"--scenes {DATA}/scenes --qa_train {DATA}/qa_train.jsonl --qa_dev {DATA}/qa_dev.jsonl --steps {STEPS} --batch {BATCH} --val_n 60 --eval_every {EV_STEP} {VLM}"
!python run.py train --variant p1 {TR} --out ../runs/p1
!python run.py train --variant p2 --prefix_only {TR} --out ../runs/p2
!python run.py train --variant p2 --resume ../runs/p2/prefix.pt {TR} --out ../runs/p2b
!python run.py train --variant p3 --resume ../runs/p2/prefix.pt --cnn_ckpts ../ck/c0_f0.pt ../ck/c0_f1.pt --gt_box_prob 0.3 --use_det {TR} --out ../runs/p3

In [ ]:
!mkdir -p ../preds
for s in ['dev', 'test']:
    EV = f"--scenes {DATA}/scenes --qa_eval {DATA}/qa_{s}.jsonl {VLM} --tta --tta_n 2"
    !python run.py eval --variant p0 {EV} --out ../preds/p0_{s}.jsonl
    !python run.py eval --variant p1 --ckpt ../runs/p1/final.pt {EV} --out ../preds/p1_{s}.jsonl
    !python run.py eval --variant p2 --ckpt ../runs/p2b/final.pt {EV} --out ../preds/p2_{s}.jsonl
    !python run.py eval --variant p3 --ckpt ../runs/p3/final.pt {EV} --out ../preds/p3_raw_{s}.jsonl
    !python run.py eval --variant p3 --ckpt ../runs/p3/final.pt {EV} --use_det --out ../preds/p3_det_{s}.jsonl
    !python run.py eval --variant p3 --ckpt ../runs/p3/final.pt {EV} --oracle_box --out ../preds/p3_oracle_{s}.jsonl
    !python run.py compare --runs P0=../preds/p0_{s}.jsonl P1=../preds/p1_{s}.jsonl P2=../preds/p2_{s}.jsonl P3-raw=../preds/p3_raw_{s}.jsonl P3=../preds/p3_det_{s}.jsonl P3-oracle=../preds/p3_oracle_{s}.jsonl --plot ../preds/p3_{s}_graph.png
from IPython.display import Image, display
display(Image(filename='../preds/p3_dev_graph.png'))